# 03 · Sequencing, evaluation, and where the LLM fits

Two questions this notebook frames (with runnable stubs) rather than fully
solves: **how do we order a playlist** when we have no audio features, and
**how do we know a generated playlist is good** with no playback telemetry?

Grounding: [`research/02`](../../research/02-music-transitions-and-flow.md),
[`research/03`](../../research/03-dj-and-radio-track-selection.md),
[`research/04`](../../research/04-measuring-playlist-success.md).

## Setup

In [ ]:
import sqlite3
import pandas as pd

DB = '../playlists.db'  # run build_db.py first if this is missing
con = sqlite3.connect(DB)
def q(sql, **kw):
    return pd.read_sql_query(sql, con, params=kw)

q('SELECT COUNT(*) AS playlists FROM playlists')

## Sequencing with metadata only
We lack tempo/key/energy (see research/02). But two craft rules from the DJ/radio
research are computable from metadata alone:
- **Artist/album separation** — don't stack the same artist back-to-back.
- **Grouping vs. scatter** — decide intentionally whether to cluster an artist or spread them.

A greedy separator: order tracks so no two adjacent share an artist (cf. the
round-robin we used for the dinner-party playlist).

In [ ]:
def separate(tracks):
    # tracks: list of dicts with 'artist'; greedy no-adjacent-repeat
    from collections import Counter, deque
    pool = list(tracks)
    out, last = [], None
    while pool:
        counts = Counter(t['artist'] for t in pool)
        # pick the most-constrained track whose artist != last
        cand = sorted(pool, key=lambda t:(t['artist']==last, -counts[t['artist']]))
        pick = cand[0]
        out.append(pick); last = pick['artist']; pool.remove(pick)
    return out

sample = q('''SELECT t.title, t.artist FROM tracks t
  JOIN playlist_tracks pt ON pt.track_id=t.id
  JOIN playlists p ON p.id=pt.playlist_id
  WHERE p.slug='this-is-daft-punk' LIMIT 40''').to_dict('records')
seq = separate(sample)
adjacent_repeats = sum(seq[i]['artist']==seq[i+1]['artist'] for i in range(len(seq)-1))
print('adjacent same-artist pairs after separation:', adjacent_repeats)

## Offline evaluation without telemetry: APC-style holdout
We can't measure skips/completion (no playback). But we can measure whether a
candidate generator **recovers held-out tracks**: hide the last k tracks of a
playlist, ask the model to complete it, score the overlap (Recall@k / R-precision).
This is exactly the RecSys 2018 / Million Playlist Dataset protocol (research/04).

In [ ]:
def holdout(slug, k=5):
    rows = q('''SELECT t.id, t.artist FROM tracks t JOIN playlist_tracks pt
                ON pt.track_id=t.id JOIN playlists p ON p.id=pt.playlist_id
                WHERE p.slug=:s ORDER BY pt.position''', s=slug)
    seed, held = rows.iloc[:-k], set(rows.iloc[-k:]['id'])
    # naive candidate gen: artists neighbouring the seed's artists (co-occurrence)
    seed_artists = tuple(seed['artist'].unique())
    ph = ','.join('?'*len(seed_artists))
    cand = pd.read_sql_query(
        f'''SELECT DISTINCT t.id FROM tracks t WHERE t.artist IN ({ph})''',
        con, params=seed_artists)
    recovered = held & set(cand['id'])
    return len(recovered)/len(held) if held else 0.0

print('Recall@5 (naive same-artist baseline):', round(holdout('this-is-oasis'),3))
# A real generator would beat this by walking co-occurrence, not just same-artist.

## Where the LLM fits
From the research, the split that plays to each tool's strength:

| Job | Best tool |
|---|---|
| Candidate generation (what songs) | co-occurrence / embeddings (data) |
| Sequencing / separation (flow) | rules + scoring (data) |
| **Theme & intent detection** ('this is a rainy-Sunday indie set') | **LLM** |
| **Naming, description, cover concept** | **LLM** |
| **Rationale / 'why these songs'** | **LLM** |
| **Quality judging vs. a rubric** (coherence, flow, fit) | **LLM-as-judge** (research/04) |

Sketch of an LLM-judge rubric call (needs `anthropic`, an API key, and the
candidate playlist as text): score 1-5 on coherence, flow, intent-fit, novelty,
with a one-line justification each. Use as an offline quality gate alongside the
holdout metric above.

See [`../../PLAN.md`](../../PLAN.md) for how these compose into a feature.